# Explore and prepare animal images

## 1. Objective
Prepare Cat, Dog, Cow, Horse and Sheep images from [Animals-10](https://www.kaggle.com/datasets/alessiocorrado99/animals10). The original ten Italian folder labels are mapped to English, then filtered to these five categories.

Pillow reads and resizes images; NumPy stores pixels; pandas organizes labels and splits; Matplotlib draws figures. ImageHash identifies candidate related images, and scikit-learn keeps their groups together when splitting.

The dataset has 26,179 images. Our five folders have 12,840 files; one confirmed label error was excluded before the original split, leaving 12,839 images. No corrupt files or exact pixel duplicates were found. Image rights and source details are recorded in `outputs/dataset_license.md`.

This notebook reuses completed preparation. Set `REBUILD_DATA` to `True` only when deliberately rebuilding the original data. It does not silently change the saved split.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / 'common.py').exists() else Path.cwd().parent
assert (ROOT / 'common.py').exists(), 'Open this notebook from the project or notebooks folder.'
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
import json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from common import LABELS, SEED, pixels
np.random.seed(SEED)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})
REBUILD_DATA = not (ROOT / "data/processed/images.npz").exists()


## 2. Understand the data
The original counts are saved in `outputs/original_class_counts.csv`. Dog is the largest selected class. Class imbalance can make accuracy look good even when Cat recall is poor.

In [ ]:
from PIL import Image, ImageOps
import hashlib, imagehash
from common import pixels
raw = ROOT / 'data/raw/raw-img'
assert raw.exists(), 'Download and unzip Animals-10 first.'
translation = {'cane':'Dog','cavallo':'Horse','elefante':'Elephant','farfalla':'Butterfly',
               'gallina':'Chicken','gatto':'Cat','mucca':'Cow','pecora':'Sheep',
               'ragno':'Spider','scoiattolo':'Squirrel'}
counts = pd.read_csv(ROOT / 'outputs/original_class_counts.csv')


## 3. Check images and exact duplicates
The preparation code fully decodes each selected image. SHA-256 detects identical files and identical oriented RGB pixels. It records corrupt files and removes exact copies if present. The saved audit keeps this evidence without repeating the scan during normal notebook use.

In [ ]:
if REBUILD_DATA:
    records, corrupt = [], []
    for directory in sorted(raw.iterdir()):
        if not directory.is_dir() or translation[directory.name] not in LABELS: continue
        for path in sorted(directory.iterdir()):
            if not path.is_file(): continue
            relative = path.relative_to(ROOT).as_posix()
            try:
                with Image.open(path) as image: image.verify()
                with Image.open(path) as image:
                    image = ImageOps.exif_transpose(image).convert('RGB'); image.load()
                    width, height = image.size
                    content = hashlib.sha256(str(image.size).encode() + image.tobytes()).hexdigest()
                    phash = str(imagehash.phash(image))
                    mirror = str(imagehash.phash(ImageOps.mirror(image)))
                records.append(dict(image_id=hashlib.sha256(relative.encode()).hexdigest()[:20],path=relative,
                    original_label=directory.name,label=translation[directory.name],width=width,height=height,
                    sha256=hashlib.sha256(path.read_bytes()).hexdigest(),pixel_sha256=content,phash=phash,mirror_phash=mirror))
            except Exception as error:
                corrupt.append({'path':relative,'reason':str(error)})
    all_images = pd.DataFrame(records)
    all_images.to_csv(ROOT / 'outputs/image_audit.csv', index=False)
    pd.DataFrame(corrupt,columns=['path','reason']).to_csv(ROOT / 'outputs/corrupt_images.csv', index=False)
    conflicts = all_images.groupby('pixel_sha256').label.nunique()
    conflict_hashes = set(conflicts[conflicts > 1].index)
    clean = all_images[~all_images.pixel_sha256.isin(conflict_hashes)].drop_duplicates('pixel_sha256').copy().reset_index(drop=True)
    removed = all_images[~all_images.image_id.isin(clean.image_id)].copy()
    removed.to_csv(ROOT / 'outputs/exact_duplicates_removed.csv', index=False)
else:
    all_images = pd.read_csv(ROOT / 'outputs/image_audit.csv')
    clean = pd.read_csv(ROOT / 'data/processed/split_manifest.csv')
    corrupt = pd.read_csv(ROOT / 'outputs/corrupt_images.csv')
    removed = pd.read_csv(ROOT / 'outputs/exact_duplicates_removed.csv')


## 4. Review related images and suspicious labels
A 64-bit perceptual hash, including horizontal reflections, identified 1,108 candidate pairs at distance six or less. Some are different photographs or false matches. We conservatively keep every detected connected group in one split.

Review covered all five cross-label pairs and 12 fixed-seed examples across hash distances. One Horse image in the Cow folder was excluded before splitting. Later review found residual label noise, including a Horse under Dog and some non-photographic images. These are recorded as limitations; final test labels were not edited after evaluation. Local contact sheets remain in `outputs/private/`.

In [ ]:
if REBUILD_DATA:
    # A small BK-tree finds nearby hashes without storing every pairwise distance.
    class HashTree:
        def __init__(self): self.root = None
        def add(self, value, index):
            if self.root is None: self.root=[value,[index],{}]; return
            node=self.root
            while True:
                distance=(value ^ node[0]).bit_count()
                if distance==0: node[1].append(index); return
                if distance not in node[2]: node[2][distance]=[value,[index],{}]; return
                node=node[2][distance]
        def query(self,value,radius):
            stack=[] if self.root is None else [self.root]
            while stack:
                node=stack.pop(); distance=(value ^ node[0]).bit_count()
                if distance<=radius:
                    for index in node[1]: yield index,distance
                stack.extend(child for d,child in node[2].items() if distance-radius<=d<=distance+radius)
    
    parent=list(range(len(clean)))
    def find(i):
        while parent[i]!=i: parent[i]=parent[parent[i]]; i=parent[i]
        return i
    def union(a,b):
        a,b=find(a),find(b)
        if a!=b: parent[max(a,b)]=min(a,b)
    tree=HashTree(); candidates=[]
    for i,row in clean.iterrows():
        matches={}
        for value in [int(row.phash,16),int(row.mirror_phash,16)]:
            for j,d in tree.query(value,6): matches[j]=min(d,matches.get(j,65))
        for j,d in matches.items():
            union(i,j)
            candidates.append({'left_id':clean.iloc[j].image_id,'right_id':row.image_id,'distance':d,
                               'left_label':clean.iloc[j].label,'right_label':row.label})
        tree.add(int(row.phash,16),i)
    clean['group_id']=[clean.iloc[find(i)].image_id for i in range(len(clean))]
    pairs=pd.DataFrame(candidates,columns=['left_id','right_id','distance','left_label','right_label'])
    pairs.to_csv(ROOT / 'outputs/near_duplicate_candidates.csv',index=False)
    # Save all candidate pairs in readable local contact sheets.
    private=ROOT / 'outputs/private'; private.mkdir(exist_ok=True)
    lookup=clean.set_index('image_id')
    for start in range(0,len(pairs),12):
        page=pairs.iloc[start:start+12]
        fig,axes=plt.subplots(len(page),2,figsize=(7,2.2*len(page)),squeeze=False)
        for axes_row,(_,pair) in zip(axes,page.iterrows()):
            for ax,key in zip(axes_row,['left_id','right_id']):
                r=lookup.loc[pair[key]]; ax.imshow(pixels(ROOT / r.path,160)); ax.axis('off')
                ax.set_title(f'{r.label} | {r.name[:8]} | distance {pair.distance}',fontsize=9)
        fig.tight_layout(); fig.savefig(private / f'near_pairs_{start//12+1:03d}.jpg',dpi=90); plt.close(fig)
else:
    pairs = pd.read_csv(ROOT / 'outputs/near_duplicate_candidates.csv')


## 5. Keep a reproducible split
The target is 70% training, 15% validation and 15% testing, with seed 42. Twenty stratified group folds assign 14 to training, three to validation and three to testing. Whole groups give actual counts of 8,986 / 1,928 / 1,925. The improved models use exactly these IDs.

Training fits the model. Validation supports model selection. The original test set has already been inspected, so later scores on it are a historical benchmark rather than an independent final assessment.

In [ ]:
if REBUILD_DATA:
    # Apply the documented visual review before constructing the final splits.
    review_path=ROOT / 'outputs/manual_review_decisions.csv'
    review=pd.read_csv(review_path)
    excluded_ids=set(review.loc[review.action.eq('exclude'),'image_id'])
    clean=clean[~clean.image_id.isin(excluded_ids)].reset_index(drop=True)
    
    from sklearn.model_selection import StratifiedGroupKFold
    splitter=StratifiedGroupKFold(n_splits=20,shuffle=True,random_state=SEED)
    folds=np.empty(len(clean),dtype=int)
    for fold,(_,idx) in enumerate(splitter.split(clean,clean.label,clean.group_id)): folds[idx]=fold
    clean['split']=np.where(folds<14,'train',np.where(folds<17,'validation','test'))
    clean['label_id']=clean.label.map({label:i for i,label in enumerate(LABELS)})
    for column in ['group_id','sha256','pixel_sha256']:
        assert clean.groupby(column).split.nunique().max()==1, f'Leakage in {column}'
    assert clean.image_id.is_unique
    clean.to_csv(ROOT / 'data/processed/split_manifest.csv',index=False)
    clean.to_csv(ROOT / 'outputs/split_manifest.csv',index=False)
    distribution=pd.crosstab(clean.label,clean.split).reindex(LABELS)
    distribution.to_csv(ROOT / 'outputs/split_class_counts.csv')
    display(distribution[['train','validation','test']])
    
    fig,ax=plt.subplots(figsize=(8,4)); distribution[['train','validation','test']].plot.bar(ax=ax)
    ax.set(ylabel='Images',xlabel='Animal',title='Retained images in each split'); ax.tick_params(axis='x',rotation=0)
    fig.tight_layout(); fig.savefig(ROOT / 'outputs/split_distribution.png',dpi=180); plt.show()
else:
    clean = pd.read_csv(ROOT / 'data/processed/split_manifest.csv')
    distribution = pd.crosstab(clean.label, clean.split).reindex(LABELS)
    display(distribution[['train','validation','test']])
    fig, ax = plt.subplots(figsize=(8,3.7))
    distribution[['train','validation','test']].plot.bar(ax=ax)
    ax.set(xlabel='Animal',ylabel='Images',title='Fixed dataset split'); ax.tick_params(axis='x',rotation=0)
    fig.tight_layout(); plt.show()


## 6. Prepare numerical inputs
The raw-pixel baselines use 24 × 24 RGB arrays for Logistic Regression and 64 × 64 for the small CNN, divided by 255. These inputs are retained for comparison.

The improved Logistic Regression models use frozen pretrained CNN features. That extractor uses the documented 224 × 224 crop and ImageNet normalization, then returns 576 numbers per image. Its pixels are processed differently from the raw baseline; the classifier still learns its own Sigmoid or Softmax decision rule from training images.

In [ ]:
if REBUILD_DATA:
    # Store compact uint8 arrays; normalize when loading them for training.
    X24=np.stack([pixels(ROOT / p,24) for p in clean.path])
    X64=np.stack([pixels(ROOT / p,64) for p in clean.path])
    np.savez_compressed(ROOT / 'data/processed/images.npz',pixels24=X24,pixels64=X64,
                        image_ids=clean.image_id.to_numpy(dtype=str))
    config={'seed':SEED,'labels':LABELS,'label_mapping':{str(i):s for i,s in enumerate(LABELS)},
            'logistic_size':[24,24],'cnn_size':[64,64],'color_mode':'RGB','orientation':'EXIF transpose',
            'resize':'Pillow bilinear, direct square resize','normalization':'float32 / 255',
            'near_duplicate_rule':'64-bit pHash distance <= 6 including horizontal reflection',
            'split_method':'StratifiedGroupKFold 20 folds; train 0-13, validation 14-16, test 17-19'}
    (ROOT / 'models/preprocessing.json').write_text(json.dumps(config,indent=2))
    summary={'original_images':int(counts.images.sum()),'selected_files':len(records)+len(corrupt),
             'corrupted':len(corrupt),'exact_removed':len(removed),'manual_excluded':len(excluded_ids),'retained':len(clean),
             'near_pairs':len(pairs),'groups':clean.group_id.nunique(),
             'split_counts':clean.split.value_counts().to_dict()}
    (ROOT / 'outputs/data_summary.json').write_text(json.dumps(summary,indent=2))
    fig,axes=plt.subplots(2,5,figsize=(11,5))
    for col,label in enumerate(LABELS):
        sample=clean[clean.label==label].sample(2,random_state=SEED)
        for row,(_,r) in enumerate(sample.iterrows()):
            axes[row,col].imshow(pixels(ROOT / r.path,160)); axes[row,col].axis('off'); axes[row,col].set_title(label)
    fig.tight_layout(); fig.savefig(ROOT / 'outputs/private/animal_samples.png',dpi=160); plt.show()
else:
    fig,axes=plt.subplots(2,5,figsize=(11,5))
    for col,label in enumerate(LABELS):
        sample=clean[clean.label==label].sample(2,random_state=SEED)
        for row,(_,r) in enumerate(sample.iterrows()):
            axes[row,col].imshow(pixels(ROOT / r.path,160)); axes[row,col].axis('off'); axes[row,col].set_title(label)
    fig.tight_layout(); fig.savefig(ROOT / 'outputs/private/animal_samples.png',dpi=160); plt.show()


## 7. What we observed
The classes are imbalanced, and sample review found label noise. Detected duplicates remain within groups, but hashing cannot find every related crop or guarantee source independence. The saved manifest, audit, review decisions and counts preserve the preparation record. No data preparation was restarted for the performance improvement.